### Q1. Embedding a query

Embed the query with the ONNX `Embedder` and check the first value of the 384-dimensional vector.

In [2]:
from embedder import Embedder

embed = Embedder()
query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)

print("shape:", v.shape)
print("len:", len(v))
print("v[0]:", v[0])
print("rounded:", round(float(v[0]), 2))

shape: (384,)
len: 384
v[0]: -0.02058203437252893
rounded: -0.02


**Answer: -0.02**

### Q2. Cosine similarity

Load the lesson pages, embed `02-vector-search/lessons/07-sqlitesearch-vector.md`, and compute the dot product with the Q1 query vector.

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print("num docs:", len(documents))

target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = next(d for d in documents if d["filename"] == target)

v_query = embed.encode(query)
v_doc = embed.encode(doc["content"])

similarity = v_query.dot(v_doc)
print("similarity:", similarity)
print("rounded:", round(float(similarity), 2))

num docs: 72
similarity: 0.36107027225589694
rounded: 0.36


**Answer: 0.37**

### Q3. Chunking and search by hand


In [4]:
import numpy as np
from gitsource import chunk_documents

# 1. Chunk the documents using the provided parameters
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Extract the content from each chunk for batch embedding
contents = [chunk["content"] for chunk in chunks]

# 3. Embed the chunks and stack them into a matrix X
X = np.array(embed.encode_batch(contents))

# 4. Score the Q1 query against all chunks (dot product)
# Note: 'v_query' is your embedded query from Q2
scores = X.dot(v_query)

# 5. Find the index of the highest-scoring chunk
best_index = np.argmax(scores)
best_chunk = chunks[best_index]

# 6. Print the result
print("Highest score:", round(float(scores[best_index]), 2))
print("Best chunk filename:", best_chunk["filename"])

Highest score: 0.65
Best chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


### Q4. Vector search with minsearch

In [5]:
from minsearch import VectorSearch

# 1. Initialize the vector search engine
vector_index = VectorSearch(keyword_fields=["filename"])

# 2. Fit the index with the embeddings matrix (X) and the chunks from Q3
vector_index.fit(X, chunks)

# 3. Embed the new query
q4_query = "What metric do we use to evaluate a search engine?"
v_q4 = embed.encode(q4_query)

# 4. Perform the search
results = vector_index.search(v_q4, num_results=1)

# 5. Extract and print the filename of the top result
print("First result filename:", results[0]["filename"])

First result filename: 04-evaluation/lessons/05-search-metrics.md


### Q5. Text search vs vector search

In [6]:
import minsearch

# 1. Set up and fit the keyword search index
# We use 'content' for the text matching and 'filename' as a keyword field
text_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

# 2. Define the new query
q5_query = "How do I store vectors in PostgreSQL?"

# 3. Perform the Keyword (Text) Search
text_results = text_index.search(q5_query, num_results=5)
text_filenames = [res["filename"] for res in text_results]

# 4. Perform the Vector Search 
# (Reusing your 'embed' model from Q1 and 'vector_index' from Q4)
v_q5 = embed.encode(q5_query)
vector_results = vector_index.search(v_q5, num_results=5)
vector_filenames = [res["filename"] for res in vector_results]

# 5. Compare the results
print("--- Top 5 Vector Search Results ---")
for f in vector_filenames:
    print(f)

print("\n--- Top 5 Text Search Results ---")
for f in text_filenames:
    print(f)

# 6. Find the specific file that is in vector results but NOT in text results
vector_only = set(vector_filenames) - set(text_filenames)
print("\n--- In Vector results, but NOT in Text results ---")
print(list(vector_only)[0] if vector_only else "None found")

--- Top 5 Vector Search Results ---
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

--- Top 5 Text Search Results ---
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

--- In Vector results, but NOT in Text results ---
02-vector-search/lessons/08-pgvector.md


### Q6. Hybrid search

In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [9]:
# 1. Define the final query
q6_query = "How do I give the model access to tools?"

# 2. Perform the Text Search
text_results_q6 = text_index.search(q6_query, num_results=5)

# 3. Perform the Vector Search
v_q6 = embed.encode(q6_query)
vector_results_q6 = vector_index.search(v_q6, num_results=5)

# 4. Fuse the results using Reciprocal Rank Fusion (RRF)
fused_results = rrf([vector_results_q6, text_results_q6])

# 5. Extract and print the top result
print("Top file after RRF:", fused_results[0]["filename"])

Top file after RRF: 01-agentic-rag/lessons/13-function-calling.md
